In [1]:
import os
from dotenv import load_dotenv

In [2]:
# Load environment variables in a file called .env

load_dotenv(override=True)
api_key = os.getenv('OPENAI_API_KEY')

if not api_key:
    print('No API key was found!')
elif not api_key.startswith('sk-proj-'):
    print('An API key was found, but it doesnt start with "sk-proj-", i.e, it is not an OpenAI API key')
elif api_key.strip() != api_key:
    print('An API key was found, but it looks like it might have space or tab characters at the start or end - please remove them.')
else:
    print('The API key found and looks good so far!')


The API key found and looks good so far!


# Google ADK

The book's first example (p. 202) grounds an agent on Google Search: retrieval is a tool the agent carries, and the model decides when to use it. As printed it only builds the agent:

| The book | Here |
|---|---|
| `model="gemini-2.0-flash-exp"` | retired (`404 NOT_FOUND`); `gemini-flash-latest` |
| no runner, no query | `InMemoryRunner` and two queries: one about an event after the model's training data, one the model can answer alone |
| — | `generate_content_config` disabling AFC, which removes a one-time genai warning (as in chapters 8, 12, 13) |

The companion notebook on GitHub writes the tool as `Google Search`, which is not valid Python; the book's `google_search` is right.

`google_search` is not a function the agent calls. Gemini runs the search on Google's side and returns the answer together with **grounding metadata**: the search queries it ran and the pages it used. The run helper prints both as `[google_search]` and `[sources]` lines, which makes the retrieval step visible. Watch the second query: no search line, because the model decided it did not need one.

The next cell holds the instruction, the queries and the run helper, with no model. The Gemini cell and the Ollama cell each build the agent and run it, so either can run on its own.

In [3]:
# Book listing: Chapter 14, Hands-On Code Example (ADK), p. 202 - Google Search.
# This cell holds the instruction, the queries and a run helper, and no model. The Gemini
# cell and the Ollama cell below each build the agent and call run_research_queries.
from google.adk.agents import Agent
from google.adk.runners import InMemoryRunner
from google.genai import types

INSTRUCTION = "You help users research topics. When asked, use the Google Search tool"

# One question the model cannot answer from its training data, one it can.
QUERIES = [
    "Who won the 2025 Nobel Prize in Physics, and for what?",
    "In one sentence, what does retrieval-augmented generation mean?",
]


async def run_research_queries(agent) -> None:
    """Runs each query in a fresh session and prints the searches, sources and answer."""
    runner = InMemoryRunner(agent=agent, app_name="research_app")
    for query in QUERIES:
        session = await runner.session_service.create_session(app_name="research_app", user_id="user_1")
        print(f"\n>>> {query}")
        message = types.Content(role="user", parts=[types.Part(text=query)])
        async for event in runner.run_async(user_id="user_1", session_id=session.id, new_message=message):
            grounding = event.grounding_metadata
            if grounding and grounding.web_search_queries:
                print(f"  [google_search] {grounding.web_search_queries}")
                sources = dict.fromkeys(chunk.web.title for chunk in grounding.grounding_chunks or [])
                print(f"  [sources] {', '.join(sources)}")
            if event.content and event.content.parts:
                for part in event.content.parts:
                    if part.text and part.text.strip():
                        print(f"--- {event.author}: {part.text.strip()}")

In [4]:
# The book's agent on Gemini. google_search is a built-in tool: Gemini runs the search
# server-side and returns the sources as grounding metadata, which the helper prints.
from google.adk.tools import google_search
from google.genai.types import GenerateContentConfig, AutomaticFunctionCallingConfig

search_agent = Agent(
    name="research_assistant",
    model="gemini-flash-latest",  # the book's gemini-2.0-flash-exp is retired
    instruction=INSTRUCTION,
    tools=[google_search],
    # ADK runs its own tool loop and never uses the genai SDK's automatic function
    # calling, but leaves it enabled, so the SDK logs a one-time warning. This removes it.
    generate_content_config=GenerateContentConfig(
        automatic_function_calling=AutomaticFunctionCallingConfig(disable=True)
    ),
)

await run_research_queries(search_agent)


>>> Who won the 2025 Nobel Prize in Physics, and for what?


  [google_search] ['2025 Nobel Prize in Physics awarded to for', 'Nobel Prize in Physics 2025 winner']
  [sources] nobelprize.org, youtube.com, cnn.com
--- research_assistant: The **2025 Nobel Prize in Physics** was awarded jointly to **John Clarke**, **Michel H. Devoret**, and **John M. Martinis**.

### Citation and Significance
They were awarded the prize:
> *"for the discovery of macroscopic quantum mechanical tunnelling and energy quantisation in an electric circuit."*

### Background
Prior to their landmark series of experiments in the 1980s, quantum phenomena—such as quantum tunneling and discrete energy levels—were typically observed only at microscopic scales (atoms, electrons, and subatomic particles). 

Clarke, Devoret, and Martinis designed superconducting electronic circuits incorporating Josephson junctions to demonstrate that macroscopic, human-scale electrical systems could also behave quantum mechanically. Their breakthroughs showed that macroscopic variables (such as e

--- research_assistant: **Retrieval-augmented generation (RAG)** is an AI framework that enhances the accuracy and reliability of large language models by retrieving relevant information from external knowledge sources to reference before generating a response.


## Google ADK + Ollama

In [5]:
# Same research agent, running on a local model served by Ollama through LiteLLM.
# Prerequisites:
#   ollama pull qwen2.5:7b
#   litellm, from the google-adk[extensions] extra (uv sync, then restart the kernel)
# Run cell 4 first (the shared cell under "# Google ADK"); this cell needs nothing else: not the Gemini cell, and no Google key.
#
# google_search is Gemini-only server-side grounding. Attaching it to any other model raises
#   ValueError: Google search tool is not supported for model ollama_chat/qwen2.5:7b
# so this agent calls a local function instead: a mock search over two canned snippets.
# Retrieval is now a tool call the model makes, visible as a [tool] line.
import os

from google.adk.models.lite_llm import LiteLlm

os.environ.setdefault("OLLAMA_API_BASE", "http://localhost:11434")
os.environ.setdefault("ADK_DISABLE_JSON_SCHEMA_FOR_FUNC_DECL", "1")
OLLAMA_MODEL = "ollama_chat/qwen2.5:7b"

SNIPPETS = {
    "nobel": "nobelprize.org: The Nobel Prize in Physics 2025 was awarded jointly to John Clarke, "
             "Michel H. Devoret and John M. Martinis for the discovery of macroscopic quantum "
             "mechanical tunnelling and energy quantisation in an electric circuit.",
    "retrieval": "Retrieval-augmented generation (RAG) retrieves relevant passages from an external "
                 "knowledge source and adds them to the prompt before the model answers.",
}


def local_search(query: str) -> str:
    """Searches the web for the query and returns the best matching snippet.

    Args:
        query: What to search for.
    """
    print(f"  [tool] local_search({query!r})")
    for keyword, snippet in SNIPPETS.items():
        if keyword in query.lower():
            return snippet
    return "No results found."


ollama_search_agent = Agent(
    name="research_assistant",
    model=LiteLlm(model=OLLAMA_MODEL),
    instruction=INSTRUCTION.replace("Google Search tool", "local_search tool"),
    tools=[local_search],
)

await run_research_queries(ollama_search_agent)


>>> Who won the 2025 Nobel Prize in Physics, and for what?


  [tool] local_search('2025 Nobel Prize in Physics winner and achievement')


--- research_assistant: The Nobel Prize in Physics 2025 was awarded jointly to John Clarke, Michel H. Devoret, and John M. Martinis for the discovery of macroscopic quantum mechanical tunnelling and energy quantisation in an electric circuit.

>>> In one sentence, what does retrieval-augmented generation mean?


--- research_assistant: Retrieval-augmented generation means using a retrieval component to fetch relevant information and augmenting a generative model to produce more accurate and context-rich outputs.


# Vertex AI RAG (Google ADK)

The book's second example (pp. 202-203) connects ADK to a Vertex AI RAG corpus: a managed index of your own documents on Google Cloud. `similarity_top_k` is how many chunks come back, `vector_distance_threshold` how far from the query a chunk may be and still count.

It is the same listing as chapter 8's cell 17, and it is construction only: the service is never handed to a `Runner` (`memory_service=`) and no agent gets a tool to query it. It needs a Google Cloud project, a corpus, credentials and the `google-adk[gcp]` extra, none of which an AI Studio key provides, so the cell checks for `RAG_CORPUS_RESOURCE_NAME` and skips without it. It calls no model, so there is no Ollama version.

In [6]:
# Book listing: Chapter 14, Hands-On Code Example (ADK), pp. 202-203 - Vertex AI RAG.
# This is a Google Cloud service, so it does NOT run on an AI Studio GOOGLE_API_KEY. It needs:
#   uv add "google-adk[gcp]"   (google-cloud-aiplatform; without it the constructor raises
#                               ImportError: The 'google-cloud-aiplatform' package is required)
#   RAG_CORPUS_RESOURCE_NAME=projects/<project>/locations/<location>/ragCorpora/<corpus-id>
#   a RAG corpus you created and filled with your own documents
#   credentials: gcloud auth application-default login
# Until RAG_CORPUS_RESOURCE_NAME is set, the cell prints what is missing and skips.
import os

RAG_CORPUS_RESOURCE_NAME = os.environ.get("RAG_CORPUS_RESOURCE_NAME")

# Define an optional parameter for the number of top similar results to retrieve.
# This controls how many relevant document chunks the RAG service will return.
SIMILARITY_TOP_K = 5

# Define an optional parameter for the vector distance threshold.
# This threshold determines the maximum semantic distance allowed for retrieved results;
# results with a distance greater than this value might be filtered out.
VECTOR_DISTANCE_THRESHOLD = 0.7

if not RAG_CORPUS_RESOURCE_NAME:
    print("Skipped: set RAG_CORPUS_RESOURCE_NAME (and install google-adk[gcp]) to run this example.")
else:
    # Import the necessary VertexAiRagMemoryService class from the google.adk.memory module.
    from google.adk.memory import VertexAiRagMemoryService

    # Initialize an instance of VertexAiRagMemoryService.
    # This sets up the connection to your Vertex AI RAG Corpus.
    # - rag_corpus: Specifies the unique identifier for your RAG Corpus.
    # - similarity_top_k: Sets the maximum number of similar results to fetch.
    # - vector_distance_threshold: Defines the similarity threshold for filtering results.
    memory_service = VertexAiRagMemoryService(
        rag_corpus=RAG_CORPUS_RESOURCE_NAME,
        similarity_top_k=SIMILARITY_TOP_K,
        vector_distance_threshold=VECTOR_DISTANCE_THRESHOLD
    )
    print(f"Connected to {RAG_CORPUS_RESOURCE_NAME}")

Skipped: set RAG_CORPUS_RESOURCE_NAME (and install google-adk[gcp]) to run this example.


# LangChain and LangGraph

The book's third example (pp. 203-205) builds the whole pipeline by hand: download a speech, split it into 500-character chunks, embed them into a vector store, and run a two-node LangGraph graph, `retrieve` then `generate`. The prompt tells the model to answer only from the retrieved context, or say it does not know. Ported:

| The book | Here |
|---|---|
| `url = "https://github.com/.../blob/master/.../state_of_the_union.txt"` | a GitHub page, so `requests` would save HTML, and now a 404. The raw file at the `v0.3` tag, saved to `generated/` |
| `TextLoader` from `langchain_community` | not installed; one `Document` built from the file, which is all the loader did |
| `from langchain.text_splitter import CharacterTextSplitter` | `langchain_text_splitters`, a separate package (added to `pyproject.toml`) |
| `from langchain_community.embeddings import OpenAIEmbeddings` | `langchain_openai`, with `text-embedding-3-small` |
| `weaviate.Client(embedded_options=EmbeddedOptions())` + `Weaviate.from_documents` | `InMemoryVectorStore` from `langchain_core`. The v3 `weaviate.Client` API is gone from `weaviate-client` 4.x, and embedded Weaviate downloads and starts a database server (not on Windows). The pattern needs only a vector index |
| `ChatOpenAI(model_name="gpt-3.5-turbo")` | `ChatOpenAI(model="gpt-4o-mini")` |
| `from langchain.schema.runnable import RunnablePassthrough` | the module is gone, and the listing never used it; dropped |
| `if __name__ == "__main__":` | collapsed |
| `print(s)` for every graph update | one line per node: each retrieved chunk's first 90 characters, then the answer. The raw update repeats all four chunks in full |

The next cell holds the data, the chunks and the graph, with no model. The graph's nodes read `retriever` and `llm` when they run, so the OpenAI cell and the Ollama cell each bind those two and run the same graph.

In [7]:
# Book listing: Chapter 14, Hands-On Code Example (LangChain), pp. 203-205.
# This cell downloads and chunks the document, defines the graph, and holds no model. The
# nodes read the globals `retriever` and `llm` when they run; the OpenAI cell and the Ollama
# cell below each bind those two with their own embeddings and model, then run the queries.
from pathlib import Path
from typing import List, TypedDict

import requests
from langchain_core.documents import Document
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.vectorstores import InMemoryVectorStore  # book: Weaviate
from langchain_text_splitters import CharacterTextSplitter  # book: langchain.text_splitter
from langgraph.graph import StateGraph, END

# --- 1. Data Preparation (Preprocessing) ---
# Load data. The book's URL is a GitHub page (HTML, not the text) and is now a 404;
# this is the raw file at the v0.3 tag. Saved under generated/ (gitignored).
url = "https://raw.githubusercontent.com/langchain-ai/langchain/v0.3/docs/docs/how_to/state_of_the_union.txt"
path = Path("generated/state_of_the_union.txt")
if not path.exists():
    path.parent.mkdir(exist_ok=True)
    path.write_text(requests.get(url, timeout=30).text)

# The book's TextLoader comes from langchain_community, not installed; it only wraps
# the file in one Document.
documents = [Document(page_content=path.read_text(), metadata={"source": str(path)})]

# Chunk documents
text_splitter = CharacterTextSplitter(chunk_size=500, chunk_overlap=50)
chunks = text_splitter.split_documents(documents)
print(f"{len(chunks)} chunks from {path}")


# --- 2. Define the State for LangGraph ---
class RAGGraphState(TypedDict):
    question: str
    documents: List[Document]
    generation: str


# --- 3. Define the Nodes (Functions) ---
def retrieve_documents_node(state: RAGGraphState) -> RAGGraphState:
    """Retrieves documents based on the user's question."""
    question = state["question"]
    documents = retriever.invoke(question)
    return {"documents": documents, "question": question, "generation": ""}


def generate_response_node(state: RAGGraphState) -> RAGGraphState:
    """Generates a response using the LLM based on retrieved documents."""
    question = state["question"]
    documents = state["documents"]

    # Prompt template from the PDF
    template = """You are an assistant for question-answering tasks.
Use the following pieces of retrieved context to answer the question.
If you don't know the answer, just say that you don't know.
Use three sentences maximum and keep the answer concise.
Question: {question}
Context: {context}
Answer:
"""
    prompt = ChatPromptTemplate.from_template(template)

    # Format the context from the documents
    context = "\n\n".join([doc.page_content for doc in documents])

    # Create the RAG chain
    rag_chain = prompt | llm | StrOutputParser()

    # Invoke the chain
    generation = rag_chain.invoke({"context": context, "question": question})
    return {"question": question, "documents": documents, "generation": generation}


# --- 4. Build the LangGraph Graph ---
workflow = StateGraph(RAGGraphState)

# Add nodes
workflow.add_node("retrieve", retrieve_documents_node)
workflow.add_node("generate", generate_response_node)

# Set the entry point
workflow.set_entry_point("retrieve")

# Add edges (transitions)
workflow.add_edge("retrieve", "generate")
workflow.add_edge("generate", END)

# Compile the graph
app = workflow.compile()


# --- 5. Run the RAG Application ---
QUERIES = [
    "What did the president say about Justice Breyer",
    "What did the president say about the economy?",
]


def run_rag_queries() -> None:
    """Streams each query through the graph, printing a short line per node.

    The book prints each raw update, which repeats all retrieved chunks in full.
    """
    for query in QUERIES:
        print(f"\n--- Running RAG Query: {query}")
        for step in app.stream({"question": query}):
            for node, update in step.items():
                if node == "retrieve":
                    print(f"[retrieve] {len(update['documents'])} chunks:")
                    for doc in update["documents"]:
                        print(f"  - {doc.page_content[:90].replace(chr(10), ' ')}...")
                else:
                    print(f"[generate] {update['generation']}")

90 chunks from generated/state_of_the_union.txt


In [8]:
# The book's pipeline on OpenAI: embed the chunks, then run both queries.
import os

from langchain_openai import ChatOpenAI, OpenAIEmbeddings  # book: langchain_community.embeddings

if not os.getenv("OPENAI_API_KEY"):
    raise RuntimeError("OPENAI_API_KEY is not set; run the Ollama cell below instead.")

# Embed and store chunks in an in-memory vector store
vectorstore = InMemoryVectorStore.from_documents(
    documents=chunks,
    embedding=OpenAIEmbeddings(model="text-embedding-3-small"),
)

# Define the retriever
retriever = vectorstore.as_retriever()

# Initialize LLM
llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)  # book: gpt-3.5-turbo

run_rag_queries()


--- Running RAG Query: What did the president say about Justice Breyer


[retrieve] 4 chunks:
  - Tonight, I’d like to honor someone who has dedicated his life to serve this country: Justi...
  - And I did that 4 days ago, when I nominated Circuit Court of Appeals Judge Ketanji Brown J...
  - A former top litigator in private practice. A former federal public defender. And from a f...
  - As I said last year, especially to our younger transgender Americans, I will always have y...


[generate] The president honored Justice Stephen Breyer for his dedication to serving the country, highlighting his background as an Army veteran and Constitutional scholar. He expressed gratitude for Breyer's service and mentioned that he nominated Judge Ketanji Brown Jackson to continue Breyer's legacy of excellence on the Supreme Court.

--- Running RAG Query: What did the president say about the economy?


[retrieve] 4 chunks:
  - And it worked. It created jobs. Lots of jobs.   In fact—our economy created over 6.5 Milli...
  - It’s time.   But with all the bright spots in our economy, record job growth and higher wa...
  - For the past 40 years we were told that if we gave tax breaks to those at the very top, th...
  - So what are we waiting for? Let’s get this done. And while you’re at it, confirm my nomine...


[generate] The president highlighted that the economy created over 6.5 million new jobs last year and grew at a rate of 5.7%, marking the strongest growth in nearly 40 years. However, he acknowledged that many families are still struggling due to inflation, which is eroding their gains. His top priority is to get prices under control while implementing a new economic vision that aims to lower costs and the deficit.


## LangGraph + Ollama

In [9]:
# Same pipeline, with local embeddings and a local model served by Ollama.
# Prerequisites:
#   ollama pull nomic-embed-text
#   ollama pull llama3.1:8b
# Run cell 11 first (the shared cell under "# LangChain and LangGraph"); this cell needs nothing else: not the OpenAI cell, and no OpenAI key.
from langchain_ollama import ChatOllama, OllamaEmbeddings

vectorstore = InMemoryVectorStore.from_documents(
    documents=chunks,
    embedding=OllamaEmbeddings(model="nomic-embed-text"),
)
retriever = vectorstore.as_retriever()
llm = ChatOllama(model="llama3.1:8b", temperature=0)

run_rag_queries()


--- Running RAG Query: What did the president say about Justice Breyer
[retrieve] 4 chunks:
  - Tonight, I’d like to honor someone who has dedicated his life to serve this country: Justi...
  - And I did that 4 days ago, when I nominated Circuit Court of Appeals Judge Ketanji Brown J...
  - But in my administration, the watchdogs have been welcomed back.   We’re going after the c...
  - Madam Speaker, Madam Vice President, our First Lady and Second Gentleman. Members of Congr...


[generate] The president nominated Circuit Court of Appeals Judge Ketanji Brown Jackson to serve on the United States Supreme Court, saying she will continue Justice Breyer's legacy of excellence. The president did not directly address Justice Breyer in the speech, but thanked him for his service. The president's nomination of Judge Jackson was made 4 days prior to the speech.

--- Running RAG Query: What did the president say about the economy?


[retrieve] 4 chunks:
  - For the past 40 years we were told that if we gave tax breaks to those at the very top, th...
  - And it worked. It created jobs. Lots of jobs.   In fact—our economy created over 6.5 Milli...
  - But in my administration, the watchdogs have been welcomed back.   We’re going after the c...
  - I understand.   I remember when my Dad had to leave our home in Scranton, Pennsylvania to ...


[generate] The president said that the economy created over 6.5 Million new jobs last year, the strongest growth in nearly 40 years, with a growth rate of 5.7%. This is a result of the president's new economic vision for America, which has brought fundamental change to an economy that hasn't worked for working people for too long. The deficit is also expected to be down to less than half what it was before the president took office by the end of the year.
